# Example queries: `report` (resstock_oedi)

Auto-generated from `tests/query_snapshots/report.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# Cache folder: resolve `tests/query_snapshots/resstock_oedi_cache` regardless
# of which directory Jupyter was launched from.
_CANDIDATES = [
    Path.cwd() / "tests/query_snapshots/resstock_oedi_cache",            # repo root
    Path.cwd() / "../query_snapshots/resstock_oedi_cache",               # tests/example_notebooks/
    Path.cwd() / "query_snapshots/resstock_oedi_cache",                  # tests/
]
_CACHE = next((p.resolve() for p in _CANDIDATES if p.exists()), _CANDIDATES[0].resolve())
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `report_buildings_by_change_ok`

report.get_buildings_by_change for upgrade 1, change_type='ok-chng' — buildings whose upgrade succeeded with an acceptable energy change. Pins the change-condition SQL shape now that the OEDI applicability fix lets the report layer resolve.


In [ ]:
result = bsq.report.get_buildings_by_change(upgrade_id='1', change_type='ok-chng')
result.head() if hasattr(result, 'head') else result


# shape: (513945, 3)
 bldg_id  applicability  applicability_1
  264875           True             True
  264963           True             True
  265162           True             True
  265209           True             True
  265252           True             True


## `report_successful_simulation_count_co`

report.get_successful_simulation_count restricted to CO. Pins the metadata-only count shape. Method was broken before fix in 2026-04-26: the call passed `bs_only=True` to _add_restrict() which doesn't accept that kwarg, causing TypeError on every call with a non-empty restrict.


In [ ]:
result = bsq.report.get_successful_simulation_count(restrict=[('state', ['CO'])])
result.head() if hasattr(result, 'head') else result


# shape: (1, 1)
 count
  9425


## `report_successful_simulation_count_no_restrict`

report.get_successful_simulation_count with no restrict — pins the no-WHERE branch (only the implicit applicability=true filter). Pairs with report_successful_simulation_count_co to cover both restrict and no-restrict paths.


In [ ]:
result = bsq.report.get_successful_simulation_count()
result.head() if hasattr(result, 'head') else result


# shape: (1, 1)
 count
549718


## `report_success_report`

report.get_success_report — returns three SQL statements (baseline counts, per-upgrade counts, change-type breakdown). Joined into one snapshot via the harness's MULTI_SQL_SEPARATOR. Pins the most-used reporting entry point. Requires the OEDI applicability fix in _rename_completed_status_column (commit added a lowercase-string normalize so True/False booleans map correctly).


In [ ]:
result = bsq.report.get_success_report(trim_missing_bs='auto')
result.head() if hasattr(result, 'head') else result


# shape: (17, 17)
 success  inapplicable  fail    Sum  Applied %  no-chng  bad-chng  ok-chng  true-bad-chng  true-ok-chng  null      any  no-chng %  bad-chng %  ok-chng %  true-ok-chng %  true-bad-chng %
  549718             0     0 549718        0.0      0.0       0.0      0.0            0.0           0.0   0.0      0.0        0.0         0.0        0.0             0.0              0.0
  528643         21075     0 549718       96.2      8.0   14690.0 513945.0        14690.0      513945.0   0.0 528643.0        0.0         2.8       97.2            97.2              2.8
  406046        143672     0 549718       73.9      2.0    5230.0 400814.0         5230.0      400814.0   0.0 406046.0        0.0         1.3       98.7            98.7              1.3
  528642         21076     0 549718       96.2      0.0    4329.0 524313.0         4329.0      524313.0   0.0 528642.0        0.0         0.8       99.2            99.2              0.8
  528641         21077     0 549718       96.2      

## `report_check_ts_bs_integrity`

report.check_ts_bs_integrity — every BuildStockQuery(skip_reports=False) init runs this. Pins the SQL shape of all queries it submits: per-upgrade ts distinct counts, metadata distinct counts (baseline + per-upgrade), duplicate-row checks, rows-per-building. Marked nondeterministic to skip data check — _get_rows_per_building scans the full TS table and would be a multi-TB landmine to actually run. SQL-shape regression coverage for the comma-join class of bug fixed in commit 9cd148d (md_table column refs mixed with bs_table predicates produced FROM x, x AS bs Cartesians that Athena rejected with 'mismatched input GROUP').


In [ ]:
result = bsq.report.check_ts_bs_integrity()
result.head() if hasattr(result, 'head') else result
